# **How Transformer LLMs Work**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Introduction](#Introduction)
  - [Course Outline](#Course-Outline)
- [Understanding Language Models: Language as a Bag-of-Words](#Understanding-Language-Models:-Language-as-a-Bag-of-Words)
  - [Understand the Landscape of Language AI Models](#Understand-the-Landscape-of-Language-AI-Models)
  - [Why Representing Language is Challenging](#Why-Representing-Language-is-Challenging)
  - [Build a Bag-of-Words Representation Step by Step](#Build-a-Bag-of-Words-Representation-Step-by-Step)
- [Understanding Language Models: (Word) Embeddings](#Understanding-Language-Models:-(Word)-Embeddings)
  - [Understand Why We Need Semantic Embeddings](#Understand-Why-We-Need-Semantic-Embeddings)
  - [Discover How Word2Vec Learns Word Meaning](#Discover-How-Word2Vec-Learns-Word-Meaning)
  - [Walk Through the Word2Vec Training Process](#Walk-Through-the-Word2Vec-Training-Process)
  - [Explore Word Embedding Dimensions and Properties](#Explore-Word-Embedding-Dimensions-and-Properties)
  - [See How Tokenization Builds Word, Sentence and Document Embeddings](#See-How-Tokenization-Builds-Word,-Sentence-and-Document-Embeddings)
- [Understanding Language Models: Encoding and Decoding Context with Attention](#Understanding-Language-Models:-Encoding-and-Decoding-Context-with-Attention)
  - [Context and RNN-Based Translation](#Context-and-RNN-Based-Translation)
  - [Walk Through the Autoregressive Translation Example](#Walk-Through-the-Autoregressive-Translation-Example)
  - [Dive Deeper into Encoding and Decoding with RNNs](#Dive-Deeper-into-Encoding-and-Decoding-with-RNNs)
  - [Understand Why Attention Improves Long-Sequence Modeling](#Understand-Why-Attention-Improves-Long-Sequence-Modeling)
  - [See How Attention Uses Hidden States for Better Outputs](#See-How-Attention-Uses-Hidden-States-for-Better-Outputs)
- [Understanding Language Models: Transformers](#Understanding-Language-Models:-Transformers)
  - [Introduction to Transformers and the Power of Attention](#Introduction-to-Transformers-and-the-Power-of-Attention)
  - [Walk Through the Encoder-Decoder Transformer](#Walk-Through-the-Encoder-Decoder-Transformer)
  - [Representation Models: Explore BERT and Masked Language Modeling](#Representation-Models:-Explore-BERT-and-Masked-Language-Modeling)
  - [Generative Models: Decoder-Only Models Like GPT](#Generative-Models:-Decoder-Only-Models-Like-GPT)
  - [Understand Context Length and Scaling Model Sizes](#Understand-Context-Length-and-Scaling-Model-Sizes)
  - [Year of Generative AI](#Year-of-Generative-AI)
- [Tokenizers](#Tokenizers)
  - [Understand Tokens and Embeddings](#Understand-Tokens-and-Embeddings)
  - [See How Text Becomes Tokens](#See-How-Text-Becomes-Tokens)
  - [Compare Word, Subword, Character, and Byte Tokens](#Compare-Word,-Subword,-Character,-and-Byte-Tokens)
  - [Tokenizing Text](#Tokenizing-Text)
  - [Visualizing Tokenization](#Visualizing-Tokenization)
    - [Optional Models to Explore](#Optional-Models-to-Explore)
- [Architectural Overview](#Architectural-Overview)
  - [Dive into the Transformer Workflow](#Dive-into-the-Transformer-Workflow)
  - [Break Down the Transformer's Core Components](#Break-Down-the-Transformer's-Core-Components)
  - [Decoding Strategies for Next-Token Choice](#Decoding-Strategies-for-Next-Token-Choice)
  - [How Parallel Processing & Caching Speeds Up Transformers](#How-Parallel-Processing-&-Caching-Speeds-Up-Transformers)
- [The Transformer Block](#The-Transformer-Block)
  - [Walk Through the Transformer Block Flow](#Walk-Through-the-Transformer-Block-Flow)
  - [Break Down the Transformer Block Components](#Break-Down-the-Transformer-Block-Components)
    - [Feed-Forward Neural Network Layer](#Feed-Forward-Neural-Network-Layer)
    - [Self-Attention Layer](#Self-Attention-Layer)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [1]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.9.6
🔢 NumPy        2.0.2
🐼 Pandas       2.3.3
📊 Matplotlib   3.9.4
🔥 PyTorch      2.8.0
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Introduction

Welcome to **How Transformer LLMs Work**. In this course, you learn about the main components of the LLM transformer architecture that has transformed the field of language processing. As you leave this course, you will be able to read through papers describing models and understand the details that are used to describe these architectures. And these intuitions will help you use LLMs better too.

Let me introduce the main topic of this course the transformer. The transformer architecture was first introduced in the 2017 paper, **Attention is All You Need** by Ashish Vaswani and others for machine translation tasks. The idea was to input an English sentence and have the network output a German sentence. The same architecture tend to be great at inputting a prompt and outputting a response to that prompt like a question and the answer to that question. And so this helped herald the early rise of large language models.

The original transformer architecture consisted of two main parts: an **Encoder** and **Decoder**. Consider translating English into German, the encoder preprocess the entire input English text to extract the context needed to perform the translation. Then the decoder uses the encoder context to generate the German. The encoder and the decoder form the basis for the models used in many language models today.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-01.png" width="800">

The encoder model provides rich, context-sensitive representations of the input text, and is the basis for the BERT model and most of the embedding models used in RAG applications. The decoder model performs text generation tasks such as summarizing text, writing code, answering questions, and is the basis for most popular LLMs, such as those from OpenAI, Anthropic, Cohere, and Meta.

## Course Outline

* You first delve into recent developments in LLMs to see how a sequence of increasingly sophisticated building blocks led to the modern transformer.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-02.png" width="800">

* You then learn about tokenization, which consists of taking text and breaking it down into tokens that comprise words or word fragments. They can then be fed into the LLM.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-03.png" width="800">

* After that, you gain intuition about how the transformer network works, focusing on the decoder-only models. A generative model takes in a text prompt, and it generates a text in response by generating one token at a time.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-04.png" width="800">

    Here's how the generation process works. The model starts by mapping each input token into an embedding vector that captures the meaning of that token. After that, the model parses these token embeddings through a stack of transformer blocks, where each block is a specific neural network architecture that is designed to learn flexibly from data and also scale well on GPUs. So you learn how each block is made up of an attention layer and a feed-forward network. The model then uses the output vectors of the transformer blocks and passes them to the last component, the language modeling head, which generates the output token.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-05.png" width="800">



The magic of LLMs actually comes from two parts: One, the transformer architecture, which is well worth learning. And second, all the incredibly rich data that the models learn from. So while the magic of LLMs comes not just from the transformer architecture, but also from the data, having a solid intuition of what this architecture is doing will give you better intuitions about why they behave in certain ways, as well as how to use them.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: Language as a Bag-of-Words

## Understand the Landscape of Language AI Models

In the next several sections, you will learn the evolution of how language has been represented numerically. We'll start with **Bag of Words**, an algorithm that represents words as large sparse vectors or arrays of numbers, which simply record the presence of words. Then **Word2Vec**, whose word representation capture the meaning of words in the context of a few neighboring words. Finally, **Transformers**, whose dense vectors capture the meaning of words in the context of a sentence or a paragraph.

Although we will explore a more recent history of language AI and language models, it is important to understand where we started. Earlier techniques such as Bag-of-Words and Word2Vec have been arguably the foundation of what we're using today. Although they lack contextualized representations, they are often a good baseline to start with. We call these **non-transformer models** since today's models are typically powered solely by transformer models, in contrast to these strong baselines.

Then we have **encoder-only models**, which are great at representing language in numerical representations. In contrast, **decoder-only models** are generative in their nature. Their main usage is to generate text. And finally, we have **encoder-decoder models** that attempt to get the best of both worlds. You will learn about these non-transformer models, encoders, decoders and how they relate, as we explore this more recent history of language AI.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-06.png" width="800">

## Why Representing Language is Challenging

Language however, is a tricky concept for computers. Text is unstructured in nature and loses its meaning when represented by zeros and ones or individual characters. As a result, throughout the history of language AI, there has been a large focus on representing language in a structured manner so that it can more easily be used by computers. From generating text, to creating numerical representations, and classifying textual inputs — these are just a few of the numerous tasks you can do with language AI.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-07.png" width="800">

## Build a Bag-of-Words Representation Step by Step

At the start of this Language AI field, the focus was mainly on representing language to analyze unstructured data. A first and still very relevant method is by representing language as a bag-of-words. Imagine you have some input text, `"That is a cute dog"`. To represent this sentence you can break it up into smaller pieces. To do so, you split the text into words by separating them through a whitespace. This process of converting the input text into pieces is called *tokenization*, and each individual word is called a token. Note that a token can be any even smaller than an entire word, but we will go through tokenization in the future sections in more detail. You can perform the same tokenization process with another document, `"My cat is cute"`. Although similar documents, they do contain different words and have a different meaning. Now that you are left with two sets of tokens, you can create something called a *vocabulary*. This vocabulary contains all unique words or tokens found in both input documents. As such, the vocabulary will contain fewer words than the amount of tokens that were generated. And we typically refer to the number of tokens or words in the vocabulary as the vocabulary size.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-08.png" width="800">

To then represent one of our inputs with numerical values, let's focus a bit on our second input, `"My cat is cute"`. This input has four tokens that match with some of the words in the vocabulary, but not all. You can then count how often a certain token appears in the vocabulary that we already created. In this case, each token appears once. However, you will also have to take note of the words in the vocabulary that do not appear in the input. A sentence not only gives meaning to the words it contains, but also the words it doesn't. Now that you have counted how often the words or tokens of our input appear in the vocabulary, you have created your numerical representation. This is called a bag-of-words, and does nothing more than counting individual words that appear in the vocabulary. Therefore, the numerical representation of `"My cat is cute"` is `0101011`, in that specific order. The order is important as it allows us to compare different sentences to one another. In practice, we call this a vector representation, a list of numerical values that represents the input. In this example, these values are counts and have an explicit meaning, namely, the number of times a word in the vocabulary appears in the input. Vector representations, in more complex and advanced models, typically do not have such an intuitive meaning, and have values between 0 and 1. In the next section, we'll learn how these vector embeddings are created.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-09.png" width="800">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: (Word) Embeddings

## Understand Why We Need Semantic Embeddings

Let's explore how these more complex vector embeddings are created. Note that bag-of-words, although an elegant approach, has a flaw. It considers language to be nothing more than an almost literal bag-of-words, and ignores the semantic nature or meaning of text.

## Discover How Word2Vec Learns Word Meaning

Released in 2013, Word2Vec was one of the first successful attempts at capturing the meaning of text in embeddings. To do so, Word2Vec learned semantic representations of words by training on vast amounts of textual data, like the entirety of Wikipedia. To generate these semantic representations, Word2Vec leverages neural networks. These networks consist of interconnected layers of nodes that process information. Neural networks can have many layers, and each connection has a certain weight depending on the inputs. These weights are often referred to as parameters of the model. Using these neural networks, Word2Vec generates word embeddings by looking at which other words they tend to appear next to in a given sentence.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-10.png" width="800">

## Walk Through the Word2Vec Training Process

You start by assigning every word in your vocabulary with a vector embedding, say, five values for each word is initialized with random values. Then in every training step, you take pairs of words from training data, and the model attempts to predict whether or not they are likely to be neighbors in a sentence. During this training process, Word2Vec learns the relationship between words and distills that information into the embedding. If the two words tend to have the same neighbor, their embeddings will be closer to one another and vice versa. The resulting embeddings capture the meaning of words.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-11.png" width="800">

## Explore Word Embedding Dimensions and Properties

Embeddings capture the meaning of words. But what exactly does that mean? To illustrate this phenomenon, let's explore an example.

Assume that you have an embedding for the word `"cats"`. This embedding has values between -1 and 1. Embeddings attempt to capture meaning by representing the properties of words. For instance, the word `"cats"` might score low on the properties `newborn` and `human` and `fruits`, while scoring high on the properties `animal` and `plural`. The number of properties or values an embedding has is called the number of dimensions, and is generally a fixed size. The same can be said for other words, such as `"puppy"`, which scores high on `animal` and `newborn`, but low on all others. By doing this for a number of words, you can use these values to get a proxy of the meaning of these words. Note that the number of dimensions can be quite large, where it is not uncommon to see embeddings with more than a thousand values. However, in practice, you do not actually know what these properties exactly represent, as they are learned through complex mathematical calculations. These properties do allow you to compare embeddings and therefore words with one another. Words with similar meaning will be grouped together, whereas different words are further apart. How similar or dissimilar certain words are, depends on the training data.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-12.png" width="800">

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-13.png" width="800">

## See How Tokenization Builds Word, Sentence and Document Embeddings

Thus far we explored word embeddings, but there are many types of embeddings that we can use. When we talk about a model like Word2Vec that converts textual input to embeddings, we refer to it as a representation model as it attempts to represent text as values. Imagine that you have some input. For example, the sentence `"Her vocalization was melodic"`. Through tokenization, you can split the sentence up into tokens. Note that this procedure is actually not splitting the input by white spaces. The word `vocalization` is split up into `vocal` and `ization`. The reason for this is that models that perform tokenization, also called tokenizers, have a fixed vocabulary. As such, they cannot represent all words that exist, but sometimes have to find combinations of words. You give the representation model these individual tokens, which in turn generates these embeddings, one for each token. Note that the word `vocalization` contains the tokens, `vocal` and `ization`. So when you average the embeddings of these tokens, you get a word embedding as it now represents the entire word. Similar techniques can be used for entire sentences to create sentence embeddings, and the same for longer texts such as documents to create a document embedding. These are just a few of the many embeddings that exist.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-14.png" width="800">

In the next section, we'll learn how you can encode and decode contextualized information, rather than static representations.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: Encoding and Decoding Context with Attention

## Context and RNN-Based Translation

In the previous sections, we explored words and token embedding techniques, but with limited contextual capabilities. In this section, we will explore how you can encode and decode context with attention.

Word2Vec create static embeddings. The same embedding is generated for the word `"bank"`, regardless of the context. `"Bank"` can refer to both a financial bank as well as the bank of a river. Its meaning, and therefore its embedding should change depending on the context. Capturing the text context is important to perform some language tasks, such as translation.

A step in encoding this text was achieved through Recurrent Neural Networks or RNNs. These are variants of neural networks that can model sequences as an additional input. To do so, these RNNs are used for two tasks, encoding or representing an input sentence, and decoding, or generating an output sentence.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-15.png" width="800">

## Walk Through the Autoregressive Translation Example

Let's explore RNN-based translation by showcasing how this input sentence `"I love llamas"` gets translated to Dutch. The text is passed to the encoder, which attempts to represent the entire sequence through embeddings. The decoder then uses those embeddings to generate language. Here it translated the English `"I love llamas"` to the Dutch `"Ik hou van lama's"`.

Each step in this architecture is autoregressive. When generating the next words, this architecture needs to consume all previously generated words. For instance, you take the input `"I love llamas"`, the model then generates the first token, which is `"Ik"`. To generate the next token, the output `"Ik"` is appended to the inputs. In step two, the input is now `"I love llamas ik"` , which in turn generates the output `"hou"`. You can continue this process at each step and continuously update the inputs with the previously generated token, until the entire output is generated. Most models are autoregressive and will therefore generate a single token each time.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-16.gif" width="800">

## Dive Deeper into Encoding and Decoding with RNNs

Let's explore this encoding and decoding in a bit more detail. You again start with the input sentence `"I love llamas"` tokenized into tokens. We can use Word2Vec to create the embeddings as the inputs. Although these embeddings are static by themselves, the encoder processes the entire sequence in one go and takes into account the context of the embeddings. The encoding step aims to represent the input as well as possible, and generates the context in the form of an embedding.

The decoder in turn is in charge of generating language, and does so by leveraging the previously generated **context embedding** to eventually generate the outputs. As we explored previously, these output tokens are generated one at a time, which is called **autoregressive**. This context embedding, however, makes it difficult to deal with longer sentences, since it is merely a single embedding representing the entire input. So the single embedding might fail to capture the entire context of a long and a complex sequence.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-17.gif" width="800">

## Understand Why Attention Improves Long-Sequence Modeling

In 2014, a solution called **Attention** was introduced that highly improved upon the original architecture. Attention allows the model to focus on parts of the input sequence that are relevant to one another, or attend to each other and amplify their signal. Attention selectively determines which words are most important in a given sentence. Take our input and output sequences, for example. Words with similar meaning like `I` and the Dutch `Ik`, which are synonyms, have higher attention weights since they are more related. `I` and `llamas` have lower attention weights since they do not relate much to each other in this particular sentence. And the same applies to all other words. By adding these attention mechanisms to the decoder step, the RNN can generate signals for each input word in the sequence related to the potential outputs.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-18.png" width="800">

## See How Attention Uses Hidden States for Better Outputs

You can again represent the input using Word2Vec embeddings and pass those to the encoder. Instead of passing only a context embedding to the decoder, the hidden states of all input words are passed to the decoder. A stateful word is an internal vector from a hidden layer of an RNN that contains the information about the previous words. The decoder then uses the attention mechanism to look at the entire sequence. Finally, this again generates the language. Due to this attention mechanism, the output tends to be much better since you now look at the entire sequence using embeddings for each token or words instead of the smaller and more limited context embedding.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-19.gif" width="800">

During generation, the model attends to the most relevant inputs. After having generated `"Ik hou van"` from using the inputs `"I love llamas"`, the attention mechanism then highlights words that are more relevant in this particular example. The sequential nature of this architecture precludes parallelization during training of the model.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-20.png" width="800">

In the next section, we'll learn how attention is used in transformer models.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: Transformers

## Introduction to Transformers and the Power of Attention

In our previous section, we explored the main techniques of Attention. In this lesson, you will explore how this technique was further developed and to this day still powers many Large Language Models. The true power of attention, and what drives the amazing abilities of most large language models, was first explored in the **Attention is All You Need** paper. This paper introduces the Transformer's architecture, which is based solely on attention without the Recurrent Neural Network (RNN). This architecture allows the model to be trained in parallel, which speeds up calculation significantly compared to the RNN based model, which does not allow parallelization.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-21.png" width="800">

## Walk Through the Encoder-Decoder Transformer

Let's explore how this transformer works. Assume you have the same input and output sequences as before. The transformer consists of stacked encoder and decoder blocks. These blocks all have the same attention mechanism that you saw previously, and by stacking these blocks, you amplify the strength of the encoder and decoder.

Let's take a closer look at the encoder. The input `"I love llamas"` is converted to embeddings, but instead of Word2Vec embeddings, we start with random values. Then **Self-attention**, which is attention focused on only the input, processes these embeddings and updates them. These updated embeddings contain more contextualized information as a result of the attention mechanism. They are passed to a feedforward neural network, which is a similar network that we explored before, to finally create contextualized token word embeddings. Remember that the encoder is meant for representing text and does a good job of generating embeddings. Self-attention is an attention mechanism, and instead of processing two separate sequences, it processes only one sequence i.e. the input sequence, by comparing it to itself.

Now, after the encoders are done processing the information, the next step is for the decoder. The decoder can take any previously generated words and pass it to the **Masked self-attention**, similar to the encoder, to process these embeddings. Intermediate embeddings are generated and passed to another attention network together with the embeddings of the encoder, thus processing both what has been generated and what you already have. This output is passed to a neural network and finally generates the next word in the sequence. Masked self-attention is similar to self-attention, but removes all values from the upper diagonal. Therefore, it masks future positions so that any given token can only attend to tokens that came before it. That helps to prevent leakage of information when generating the output.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-22.gif" width="800">

## Representation Models: Explore BERT and Masked Language Modeling

The original transformer model is an encoder-decoder architecture that serves translation tasks well, but cannot easily be used for other tasks like text classification. In 2018, a new architecture called **Bidirectional Encoder Representations from Transformers or BERT** was introduced that could be leveraged for wide variety of tasks.

BERT is an encoder only architecture that focuses on representing language and generating contextual word embeddings. These encoder blocks are the same as we saw before, self-attention followed by feed-forward neural networks. The input contains an additional token, the **CLS or classification token**, which is used as a representation for the entire input. Often we use this CLS token as the input embedding for fine-tuning the model on specific tasks like classification.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-23.gif" width="800">

To train a BERT-like model, you can use a technique called **Masked Language Modeling**. You first randomly mask a number of words from your input sequence and have the model predict these masked words. By doing so, the model learns to represent language as it attempts to deconstruct these masked words. Training is typically a two-step approach. First, you apply masked language modeling on large amounts of data, and this is called pre-training. After which you can fine-tune your pre-trained model on a number of downstream tasks, including classification.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-24.gif" width="800">

## Generative Models: Decoder-Only Models Like GPT

Generative models, in contrast, use a different architecture. Assume that you again have another input sequence and randomly initialized embeddings. The input is then passed to the decoder only, as generative models tend to only stack decoders. One of the first implementations is called **GPT or GPT-1**. It stands for **Generative Pre-Trained Transformer** as it uses the deep transformer decoder. A decoder block uses again masked self-attention, which is then passed to a neural network. Note that it **does not use any encoders** as we explored previously. And finally, the next word is generated.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-25.gif" width="800">

These are the two flavors that you will see most often: Generative models like ChatGPT, and Representation models like BERT or other embedding models.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-26.png" width="800">

## Understand Context Length and Scaling Model Sizes

These models have something in common called the **Context Length**. You start from an input sequence `"Tell me something about llamas"` that we ask of the generative model in this example. Now let's say that you already generated some tokens previously. The original query, together with the previously generated tokens, represent the current context length. That is, the amount of tokens that are currently being processed. In contrast, a generative LLM like GPT-1 or even a representation model can have a maximum context length, for example, 512. That means that the model can only process 512 tokens at a given time. Note that this also includes the tokens that are being generated as they update the current context length.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-27.gif" width="800">

These generative models do the Large Language Models justice. GPT-1 already had more than **100 million** parameters. The next version, GPT-2, had over **1 billion** parameters, and GPT-3 with already **175 billion** parameters. As the number of parameters grew, so did their capabilities. That is why you will often see such large models.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-28.gif" width="800">

## Year of Generative AI

Looking back at an year that we playfully called the Year of Generative AI. It all started with the well-known **ChatGPT** model or more accurately, **GPT 3.5**. Following the success of ChatGPT, many other **proprietary models** soon followed. Fortunately, **open-source models** followed quickly. These are models that have their weights publicly available for us to use. Some of them can even be freely used for commercial purposes.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-29.gif" width="800">

Let's go to the next lesson and learn about tokens and embeddings.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Tokenizers

## Understand Tokens and Embeddings

In the last lesson, you learned how word meaning was represented in vectors. We briefly introduced tokens as words or word pieces. In this lesson, we will illustrate what those are and how they help transformers do their jobs.

Imagine you have a given input sentence like `"Have the bards who"`. For a language model to process that input text, it will first break down the text into smaller pieces. Each piece is called a *token*. This process of breaking down the text is called **Tokenization**. Each token is then turned into a numerical representation, also called *embeddings*. These are vector values that represent the semantic nature of a given text. These embeddings are static, and each embedding is created independent from all other embeddings and tokens. These embeddings are processed by the large language model and converted into *contextualized embeddings*. These contextualized embeddings are still one for each input token, but has been processed such that all other tokens are considered. These embeddings can be either the output of a model (representation models like BERT) or can also be used by the model to then create the outputs (generative models like GPT). In the case of generative models, the output can be another token.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-30.gif" width="800">

## See How Text Becomes Tokens

Let's explore in detail how this tokenization process works. Given an input sentence, `"Have the bards who"`, it's tokenized, or encoded into smaller pieces. Tokens can be entire words or pieces of a word. When these pieces are combined, they form the original words. This process is necessary as tokenizers have a limited number of tokens or vocabulary, so whenever it encounters an unknown word, it can still be represented by these sub tokens. Each token has an associated fixed ID to easily encode and decode the tokens. These are fed to the language model that internally creates the token embeddings. The output of a generative model would then be another token ID, which is decoded to represent an actual token or word.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-31.gif" width="800">

## Compare Word, Subword, Character, and Byte Tokens

There are many different tokenization levels that we can explore. Assume you have an input: `"Have the bards who preceded..."`, together, with a special symbol.

When you represent this input as *word tokens*, the entire sequence is represented by words. *Subword tokens*, the example that we saw previously, can either be the entire word itself, or some pieces of the original word. If the tokenizer does not have a token for `bards`, it can still represent it with its tokens for `B` and `ards`.

You can split subword tokens into *character tokens*, one for each character in the original word, representing the entire input as nothing more than each individual character. The smallest representation is that of *byte tokens*, which is used to encode a single character of text in a computer. This is done for every single character. Note how the symbol needs additional bytes for representation, as that symbol is more complex than just a single-character representation.

In practice, most large language models have tokenizers that work on a subword token level. Its vocabulary is flexible and allows to represent most words by either its entire representation or using subtokens.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-32.gif" width="800">

## Tokenizing Text

In this section, you will tokenize the sentence `"Hello World!"` using the tokenizer of the [`bert-base-cased` model](https://huggingface.co/google-bert/bert-base-cased).

In [2]:
!pip install transformers

Let's import the `AutoTokenizer` class, define the sentence to tokenize, and instantiate the tokenizer.

**FYI**: The transformers library has a set of Auto classes, like `AutoConfig`, `AutoModel`, and `AutoTokenizer`. The Auto classes are designed to automatically do the job for you.

In [13]:
from transformers import AutoTokenizer

# Define the sentence to tokenize
sentence = "Hello world!"

# Load the pretrained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

You'll now apply the tokenizer to the sentence. The tokeziner splits the sentence into tokens and returns the IDs of each token.

In [14]:
# Apply the tokenizer to the sentence and extract token ids
token_ids = tokenizer(sentence).input_ids

print(f'token_ids: {token_ids}')

token_ids: [101, 8667, 1362, 106, 102]


To map each token ID to its corresponding token, you can use the `decode` method of the tokenizer.

In [15]:
for token_id in token_ids:
    print(tokenizer.decode(token_id))

[CLS]
Hello
world
!
[SEP]


Note how you now see five tokens represented. The sentence `"Hello world!"` starts with the CLS token. The **CLS token or the classification token** that we explored previously represents the entire input. Then you have tokens for `Hello`,  `world` and `!`. And there's also the special character SEP. The **SEP token or the separation token** signifies the end of a sentence.

## Visualizing Tokenization

Here, we wrap the code of the previous section in the function `show_tokens`. The function takes in a text and the model name, and prints the vocabulary length of the tokenizer and a colored list of the tokens.

In [16]:
# A list of colors in RGB
colors = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence: str, tokenizer_name: str):
    """
    Show the tokens each separated by a different color
    """
    # Load the tokenizer and tokenize the input
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids

    # Extract vocabulary length
    print(f"Vocab length: {len(tokenizer)}")

    # Print a colored list of tokens
    for idx, token_id in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors[idx % len(colors)]}m' +
            tokenizer.decode(token_id) +
            '\x1b[0m',
            end=' '
        )

Here's the text that you'll use to explore the different tokenization strategies of each model.

In [17]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

This text contains all different types of words to showcase how these tokenizers will process this particular input. Note that we have words that are fully capitalized, we have values in there, and that we even have special symbols to see what the tokenizer will do when faced with this particular sequence.

**bert-base-cased**

In [18]:
show_tokens(text, "bert-base-cased")

Vocab length: 28996
[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

The vocabulary length of this tokenizer shows a value of almost 30,000, which means that this tokenizer has almost 30,000 values or tokens that it can represent. Below that you will see all the tokens that it used for representing your input text. As we explored before, it starts with the CLS token, the classification token to showcase or to represent the entire input. This tokenizer has a lot of difficulty representing the word `CAPITALIZATION`. Note how it has to break that down into many tokens before it can actually represent the words. Those hashtags (`##`) showcase that this token belongs to the token before that, and that together they represent a single word. You will also see the `UNK` token appearing up in these tokens. It's the unknown token and is used when it simply doesn't know how to represent any token.

You can also try the uncased version of the bert model, and compare the vocab length and tokenization strategy of the two BERT versions.

**bert-base-uncased**

In [19]:
show_tokens(text, "bert-base-uncased")

Vocab length: 30522
[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

Next, let's see how a more recent tokenizer differs from this BERT model. We will this time use the **GPT-4 tokenizer**. It's called **Tiktoken**, and although we cannot access the GPT-4 model itself because it's proprietary, the tokenizer is still available and you can run this.

**GPT-4 (tiktoken)**

In [21]:
show_tokens(text, "Xenova/gpt-4")

Vocab length: 100263

 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         "
 12 . 0 * 50 = 600 
 

What you will see is a vocabulary length that's more than three times as large as the BERT model that you explored previously, with 100,000 tokens. It needs to use fewer tokens to represent the inputs. `CAPITALIZATION` is now only two tokens. Since this tokenizer is meant for a generative representation, it doesn't have the CLS and the SEP classification tokens or those special characters. It's interesting to see how it needs fewer tokens to represent its entire inputs. In fact, it represents even the tabs very well.

Because this model has such a large vocabulary length, it's easier to represent uncommon tokens, but there's a trade-off. The larger the vocabulary length is, the more embeddings need to be calculated for each of those tokens. So there's a trade-off between choosing a tokenizer with potentially a length of a million tokens, and actually learning the representations for each of those tokens.

Now, many of the models you can find online on the **HuggingFace** platform. There you will find much of the tokenizers to use.

### Optional Models to Explore

You can also explore the tokenization strategy of other models. The following is a suggested list. Make sure to consider the following features when you're doing your comparison:

* Vocabulary length
* Special tokens
* Tokenization of the tabs, special characters and special keywords

**GPT-2**

In [23]:
show_tokens(text, "gpt2")

Vocab length: 50257

 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

**Flan-T5-small**

In [26]:
show_tokens(text, "google/flan-t5-small")

Vocab length: 32100
English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600  </s> 

**Starcoder 2 - 15B**

In [28]:
show_tokens(text, "bigcode/starcoder2-15b")

Vocab length: 49152

 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

**Phi-3**

In [30]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

Vocab length: 32011
 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

**Qwen2 - Vision-Language Model**

In [32]:
show_tokens(text, "Qwen/Qwen2-VL-7B-Instruct")

Vocab length: 151657

 English  and  CAPITAL IZATION 
 🎵  � � � 
 show _tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         "
 1 2 . 0 * 5 0 = 6 0 0 
 

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Architectural Overview

## Dive into the Transformer Workflow

So far you have learned how language is represented numerically and how words are converted into tokens and dense embeddings. Now you're ready to dive into the details of the transformer. When we're thinking about transformer large language models, we know that there's an input prompt to the model. And then there's the generation where the model generates an output text. Now one of the first important intuitions about how the transformer works is to understand that it generates tokens one by one. So to generate that output, you can see that the model is generating them one token at a time.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-33.gif" width="800">

## Break Down the Transformer's Core Components

We'll break down these generation steps so you can understand the underlying mechanisms of how this works. The transformer is made up of three major components:
* The **tokenizer** is the component that breaks down the text into multiple chunks.
* The output of that tokenizer goes to a **stack of transformer blocks**. This is where the majority or the vast majority of computing is. These are the neural networks that do pretty much all the magic that happens here.
* The output of this stack of transformer blocks goes into a neural network called the **Language Modeling head**.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-34.gif" width="800">

Let's look at each of these components. We looked at the tokenizer, and we know by now that the tokenizer has a vocabulary that breaks down text to independent chunks. Let's say we have this vocabulary of 50,000 tokens that this tokenizer knows. The model has associated token embeddings for each of these tokens. So if we have 50,000 tokens in our vocabulary, the model has 50,000 vectors that represent each of these tokens. And these are substituted in the beginning once the model is processing its inputs, as we'll see. We'll talk about transformer blocks in the next lesson. But let's go over the language modeling head. What happens at the very end of the processing of a language model is a kind of scoring or a token probability calculation based on all of the processing done by the model in the stack of transformers, to make sense of the input context and what is requested, let's say, in the prompt and what the next token should be in response to that. And so the result of the language modeling head is this token probability scores. The probability of each of these tokens have to add up to 100%. So if you have the word `"Dear"` scored at 40% probability, that becomes the highest probability token. It can be the next output token, if you the highest probability token.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-35.gif" width="800">

## Decoding Strategies for Next-Token Choice

There are different strategies to choose the next token and are called Decoding strategies. If you're choosing the top-scoring token all the time, it's a good strategy for a lot of cases. It is what happens when you set the temperature parameter to zero. It is a method called **Greedy decoding**. But it's not the only method out there. There are others like **Top P decoding** for example, that incorporate multiple tokens. So it might generate "Dear". But it also in some cases with lower probability, might pick the next highest probability token. It definitely looks at scoring, but it doesn't always have to pick the top one. And these are sometimes important to generate text that sounds natural. And this is why you get two different answers, when you generate multiple times using the same prompt, especially if you set the temperature to values more than zero.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-36.gif" width="800">

## How Parallel Processing & Caching Speeds Up Transformers

Another important idea or intuition that make transformers operate a lot better than previous methods like RNNs, is that they **process all of their input tokens in parallel**, and that parallelization makes it time efficient. So we can compute a long context on a lot of different GPU cores in same time. The way to envision this is to think of multiple tracks flowing through this stack of transformer blocks. The number of tracks here is the context size of the model. So if a model, for example, has a context size of 16,000 tokens, it can process 16,000 tokens at the same time. The generated token in the decoder is the output of the final token in the prompt. Generating every other token is a little bit different. Once we generate our first token, now we feed that entire prompt with the token into the transformer again. You can think of it as a loop. And then where you process the inputs one by one as you generate these tokens, one thing that is different between that first step and the second step is that you can cache these calculations because they're going to be exactly the same calculation. You can cache them to speed up the generation of the model. This is what is called **KV caching**. **Time to First Token (TTFT)** is the duration between sending a prompt to a Large Language Model (LLM) and receiving the very first token. And then generating every other token is a slightly different process.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-37.gif" width="800">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# The Transformer Block

## Walk Through the Transformer Block Flow

In the architectural overview of the transformer, you have learned that after tokenization, the token embeddings are passed through a stack of transformer blocks. Let's get into the details of those blocks.

Let's look at a basic example. Let's say we have the prompt, these two words `"The Shawshank"`. To understand how transformer blocks operate, let's think about the two tracks that are flowing through the stack of transformer blocks. In the beginning, our tokenizer has broken down the prompt into these two tokens - `"The"` and `"Shawshank"`. And then since we have the associated embedding vectors for each of these, we just replace them. Now we have turned language into numbers, and we can apply a lot of interesting calculus on them to predict what the next word is. So those two embeddings flow to the first transformer block which generates a vector of the same size as its input. But something has happened in the middle. There's some processing that has happened in the transformer block in its components. But before we get into that understanding, this general flow through the model is useful. And so the same thing happens now with the second transformer block that operates on the outputs of the first transformer block in parallel across the various tracks. And this happens down the list of of transformer blocks all the way to the end. And then in the final layer, the output vector for the final token in the prompt is presented to the language modeling head, which then outputs or generates the predicted next token. So this is the flow.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-38.png" width="800">

So everything flows from the beginning towards the end, one direction from the tokenizer down the transformer blocks one by one in sequence up until the language modeling head.

## Break Down the Transformer Block Components

The transformer block itself is made up of two major components:

* the **Self-Attention** layer
* the **Feed-Forward Neural Network** layer

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-39.png" width="800">

### Feed-Forward Neural Network Layer

We can now have a high-level intuition of the feed-forward neural network. If the transformer block only had this and it didn't have attention, it would still be able to generate the next token, i.e. to say the token that is most probable to come after `"The Shawshank"` would be `"Redemption"`. This is in reference to the famous film, because if you look at the training datasets, if you get data from the internet or Wikipedia, `"Redemption"` would be the word that often statistically appears after `"Shawshank"`. And in most cases, this is one thing that the feedforward neural network is able to do. So you can think of it as a storage information and statistics of the next word that comes in after the input token. That's a high level intuition.

Like a lot of these neural networks do, this is a reasonable first approach of how to think about the feed-forward neural network. And this contrasts with self-attention. Neural networks generally have a layer that expands into another layer, and then shrinks back down into third or output layer. The connections of these dense layers is presumably where all of this information that models know is stored, modeled, interpreted and interpolated to enable the models to do the incredible things that they do to generate code and encode information about the world and speak to you in fluent and coherent language of your choosing.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-40.gif" width="800">

### Self-Attention Layer

Self-attention, on the other hand, allows the model to attend to previous tokens and incorporate the context in its understanding of the token it's currently looking at. Let's say we have a prompt that is `"The dog chased the llama because it"`. When the model is processing the word `"it"`, it needs to bake in some information about what does `"it"` refer to here. Is it the dog or is it the llama? It's important for the model to be able to have some sense of what that `"it"` refers to. And that's what self-attention does. For example, if the previous context points towards this being the `"llama"`, self-attention allows the model to bake in some of that representation of the `"llama"` into it. And so while it's processing the token, it has some understanding that this is referring to the `"llama"`. This is an NLP task called **Co-reference Resolution**.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-41.png" width="800">


To understand attention at a high level, let's formulate it like this. We have previous tokens that we've processed in the past. And then we have the current position that we are processing. We have its vector representation. This is the embedding if we're in transformer on block number one. But if you're in block number 2 or 3 or 4, this is the output of the previous block. And then towards the end, we want this vector representation that bakes in the relevant information from the other positions here.

Here you're looking at let's say five tracks. What self-attention does is two things:
* **Relevance Scoring**: it assigns a score to how relevant each of the input tokens are to the token we're currently processing.
* **Combining Information**: it combines the relevant information (from relevant tokens) into the representation of current vector.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-42.gif" width="800">

In the next lesson, we'll look at more details of how that is done.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[DeepLearning.AI: How Transformer LLMs Work Course](https://www.deeplearning.ai/courses/how-transformer-llms-work)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)